# Laboratorio 9 — Embeddings

**Milton Beltrán** · Natural Language Processing

**Corpus:** *Spanish News Classification* (Kaggle: [`kevinmorgado/spanish-news-classification`](https://www.kaggle.com/datasets/kevinmorgado/spanish-news-classification)) y *El ingenioso hidalgo don Quijote de la Mancha* (Kaggle: [`manuelmaaf97/quijote`](https://www.kaggle.com/datasets/manuelmaaf97/quijote)).
**Vectores preentrenados:** fastText SUC, tamaño M, 100 dimensiones ([dccuchile/spanish-word-embeddings](https://github.com/dccuchile/spanish-word-embeddings/blob/master/emb-from-suc.md)).

Volvemos al corpus de noticias de los laboratorios #1 a #4, pero tratado como en el #5, porque ahora lo que importa es el orden de las palabras. Word2vec aprende de las ventanas de vecinos, así que el texto se prepara como una lista de oraciones y no como una bolsa de palabras.
El Quijote se usa como segundo corpus para ver cómo cambian los vecinos de una misma palabra según el texto con el que se entrena.

| Labs #1–#4 (clasificación) | Lab #9 (embeddings) |
|---|---|
| Se eliminan las *stopwords* | Se conservan, porque son parte de la ventana de contexto |
| Stemming con `SnowballStemmer` | Ninguno, "gobiernos" y "gobernó" tienen que tener vectores distintos |
| Bolsa de palabras por documento | Lista de oraciones con los tokens en orden |
| Puntuación eliminada | También se elimina, si no ocuparía lugares de la ventana |

El archivo de fastText SUC pesa 1.1 GB y no se sube al repositorio. Hay que descargarlo de [Zenodo](https://zenodo.org/record/3234051/files/embeddings-m-model.vec?download=1) y dejarlo como `embeddings-m-model.vec` en la misma carpeta que este notebook.

## 0. Preparación del entorno y carga de los corpus

In [131]:
# --- 0.1 Librerías ---
# La variable de NLTK va antes del import: su hook de seguridad bloquea `regex` cuando
# el .venv vive dentro del directorio de trabajo.
import os

os.environ["NLTK_DISABLE_IMPORT_SECURITY"] = "1"

import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
import kagglehub

import gensim
from gensim.models import KeyedVectors
import sklearn
import torch

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("NLTK", nltk.__version__, "· gensim", gensim.__version__,
      "· scikit-learn", sklearn.__version__, "· PyTorch", torch.__version__)

NLTK 3.10.1 · gensim 4.4.0 · scikit-learn 1.7.2 · PyTorch 2.14.1


In [132]:
# --- 0.2 Recursos de NLTK ---
for recurso in ["punkt", "punkt_tab"]:
    nltk.download(recurso, quiet=True)

print("Recursos de NLTK listos.")

Recursos de NLTK listos.


In [133]:
# --- 0.3 Corpus de noticias ---
# Misma base de los labs #1–#4: 1,142 documentos tras quitar los duplicados exactos.
df = pd.read_csv("df_total.csv").drop_duplicates().reset_index(drop=True)

print(f"Documentos: {len(df):,} · Categorías: {df['Type'].nunique()}")
df.head(3)

Documentos: 1,142 · Categorías: 7


,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas


In [134]:
# --- 0.4 Quijote: descarga y recorte del aparato editorial ---
# Las mismas celdas 0.3–0.7 del Lab #5, condensadas.
ruta_txt = Path(kagglehub.dataset_download("manuelmaaf97/quijote")) / "don-quijote.txt"
crudo = ruta_txt.read_text(encoding="utf-8-sig")  # el archivo trae BOM

ini = crudo.index("*** START OF THIS PROJECT GUTENBERG EBOOK")
ini = crudo.index("\n", ini) + 1
fin = crudo.index("*** END OF THIS PROJECT GUTENBERG EBOOK")
libro = crudo[ini:fin].strip()
libro = libro[libro.index("El ingenioso hidalgo don Quijote de la Mancha"):
              libro.rindex("\nFin\n")].strip()

print(f"Cuerpo del libro: {len(libro):,} caracteres")

Cuerpo del libro: 2,097,945 caracteres


In [135]:
# --- 0.5 Quijote: párrafos y encabezados ---
# Wrap duro a ~76 columnas: hay que rearmar los párrafos antes de sentenizar.
# Los encabezados de capítulo se quitan; cada uno cuenta como un documento.
bloques = re.split(r"\n\s*\n", libro)
parrafos = [re.sub(r"\s+", " ", " ".join(b.split("\n")).strip()) for b in bloques]
parrafos = [p for p in parrafos if p]

ENCABEZADO = re.compile(r"^(Cap[íi]tulo\b|(Primera|Segunda|Tercera|Cuarta) parte del ingenioso)")
n_capitulos = sum(1 for p in parrafos if re.match(r"^Cap[íi]tulo\b", p))
parrafos_quijote = [p for p in parrafos if not ENCABEZADO.match(p)]

print(f"Capítulos: {n_capitulos} · Párrafos de prosa: {len(parrafos_quijote):,}")

Capítulos: 126 · Párrafos de prosa: 5,060


## 1. Preparación de los corpus

Pasamos del texto a una lista de oraciones, cada una con sus tokens en orden, que es el formato que espera gensim.

In [136]:
# --- 1.1 Normalización ---
# Minúsculas, sin puntuación, con tildes y números. Sin stopwords ni stemming (D-02).
LETRA = r"A-Za-zÁÉÍÓÚÜÑáéíóúüñ"
RAYA = re.compile(rf"(?<![{LETRA}])-|-(?![{LETRA}])")
APERTURA = re.compile(r"([¿¡«»])")
# "imaginamos.Para": párrafos pegados sin espacio en las noticias (P-01).
PEGADO = re.compile(r"([^\s][.!?])([A-ZÁÉÍÓÚÑ][a-záéíóúñ]|[¿¡])")
EEUU = re.compile(r"EE\s*\.\s*UU\b\.?")
MILES = re.compile(r"(?<=\d)\.(?=\d{3}\b)")  # SUC escribe "10000", no "10.000"
ALFANUM = re.compile(r"[a-záéíóúüñ0-9]")


def limpia(parrafo):
    parrafo = MILES.sub("", EEUU.sub("EEUU", parrafo))
    parrafo = PEGADO.sub(r"\1 \2", parrafo)
    return APERTURA.sub(r" \1 ", RAYA.sub(" - ", parrafo))


def tokeniza(oracion):
    """Minúsculas y solo tokens con alguna letra o dígito."""
    tokens = word_tokenize(oracion.lower(), language="spanish")
    return [t.strip(".…") for t in tokens if ALFANUM.search(t)]


demo = "Yo creo que el no tener un concepto de negocio sostenible puede tener un mayor impacto de lo que imaginamos.Para García el reto en EE.UU. es otro: 18.000 empleos."
print(sent_tokenize(demo, language="spanish"))
print([tokeniza(o) for o in sent_tokenize(limpia(demo), language="spanish")])

['Yo creo que el no tener un concepto de negocio sostenible puede tener un mayor impacto de lo que imaginamos.Para García el reto en EE.UU. es otro: 18.000 empleos.']
[['yo', 'creo', 'que', 'el', 'no', 'tener', 'un', 'concepto', 'de', 'negocio', 'sostenible', 'puede', 'tener', 'un', 'mayor', 'impacto', 'de', 'lo', 'que', 'imaginamos'], ['para', 'garcía', 'el', 'reto', 'en', 'eeuu', 'es', 'otro', '18000', 'empleos']]


In [137]:
# --- 1.2 Segmentación en oraciones y tokenización ---
def a_oraciones(textos):
    oraciones = []
    for texto in textos:
        for parrafo in texto.split("\n"):
            parrafo = limpia(parrafo.strip())
            if not parrafo:
                continue
            for o in sent_tokenize(parrafo, language="spanish"):
                tokens = tokeniza(o)
                if tokens:
                    oraciones.append(tokens)
    return oraciones


oraciones_noticias = a_oraciones(df["news"])
oraciones_quijote = a_oraciones(parrafos_quijote)

print(f"Noticias: {len(oraciones_noticias):,} oraciones")
print(f"Quijote:  {len(oraciones_quijote):,} oraciones")
print()
print(oraciones_noticias[5])
print(oraciones_quijote[100])

Noticias: 20,909 oraciones
Quijote:  10,585 oraciones

['el', 'directivo', 'argumentó', 'que', 'lo', 'que', 'se', 'encuentra', 'en', 'juego', 'acá', 'no', 'es', 'un', 'tema', 'de', 'rentabilidad', 'o', 'de', 'negocios', 'en', 'particular', 'es', 'un', 'tema', 'de', 'viabilidad', 'del', 'mundo', 'de', 'los', 'negocios', 'y', 'del', 'mundo', 'físico', 'en', 'general', 'como', 'lo', 'conocemos']
['soneto']


In [ ]:
# --- 1.3 ¿SUC distingue mayúsculas y tildes? ---
# Se carga una sola vez; el resto del laboratorio reutiliza `suc`.
t0 = time.time()
suc = KeyedVectors.load_word2vec_format("embeddings-m-model.vec", limit=200_000)
print(f"SUC: {len(suc.key_to_index):,} palabras × {suc.vector_size} dims ({time.time() - t0:.0f} s)\n")

consultas = ["españa", "España", "espana", "gobierno", "Gobierno", "mañana", "manana",
             "eeuu", "2019", "10.000", "10000", ",", "¿"]
pd.DataFrame({
    "palabra": consultas,
    "en SUC": [w in suc.key_to_index for w in consultas],
    "rango": [suc.key_to_index.get(w, "—") for w in consultas],
})

In [ ]:
# --- 1.3 (cont.) Palabras con mayúscula en todo el vocabulario cargado ---
con_mayuscula = [w for w in suc.index_to_key if w != w.lower()]
print(f"Palabras con alguna mayúscula: {len(con_mayuscula)} de {len(suc.index_to_key):,}")
print("Las 10 más frecuentes de SUC:", suc.index_to_key[:10])

Palabras con alguna mayúscula: 0 de 200,000
Las 10 más frecuentes de SUC: ['</s>', '.', 'de', ',', 'la', 'que', 'el', 'en', 'y', 'a']


In [ ]:
# --- 1.4 Tabla de los corpus ---
def resumen(nombre, documentos, oraciones):
    conteo = Counter(t for o in oraciones for t in o)
    n_tokens = sum(conteo.values())
    return {"corpus": nombre, "documentos": documentos, "oraciones": len(oraciones),
            "tokens": n_tokens, "tipos": len(conteo),
            "tokens por oración": round(n_tokens / len(oraciones), 1)}


tabla_corpus = pd.DataFrame([
    resumen("Noticias", len(df), oraciones_noticias),
    resumen("Quijote (capítulos)", n_capitulos, oraciones_quijote),
]).set_index("corpus")
tabla_corpus

,documentos,oraciones,tokens,tipos,tokens por oración
corpus,,,,,
Noticias,1142,20909,605523,30750,29.0
Quijote (capítulos),126,10585,378666,22842,35.8


In [ ]:
# --- 1.5 Vocabulario según min_count ---
filas = []
for nombre, oraciones in [("Noticias", oraciones_noticias), ("Quijote", oraciones_quijote)]:
    conteo = Counter(t for o in oraciones for t in o)
    n_tokens = sum(conteo.values())
    for k in (1, 2, 5):
        fuera = sum(f for f in conteo.values() if f < k)
        filas.append({"corpus": nombre, "min_count": k,
                      "vocabulario": sum(1 for f in conteo.values() if f >= k),
                      "% tokens fuera": round(100 * fuera / n_tokens, 2)})

pd.DataFrame(filas).set_index(["corpus", "min_count"])

vocabulario  % tokens fuera
corpus   min_count                             
Noticias 1                30750            0.00
         2                16832            2.30
         5                 8595            5.91
Quijote  1                22842            0.00
         2                11690            2.95
         5                 5275            7.40

In [ ]:
# --- 1.6 Pares skip-gram a mano, ventana 2 ---
def pares_skipgram(tokens, ventana):
    pares = []
    for i, central in enumerate(tokens):
        for j in range(max(0, i - ventana), min(len(tokens), i + ventana + 1)):
            if j != i:
                pares.append((central, tokens[j]))
    return pares


oracion = next(o for o in oraciones_noticias if " ".join(o) == "nunca pensé que pudiera trabajar en un banco")
pares = pares_skipgram(oracion, ventana=2)

print(" ".join(oracion), f"→ {len(oracion)} tokens, {len(pares)} pares\n")
pd.DataFrame(pares, columns=["central", "contexto"]).groupby("central", sort=False)["contexto"].apply(list)

nunca pensé que pudiera trabajar en un banco → 8 tokens, 26 pares



central
nunca                            [pensé, que]
pensé                   [nunca, que, pudiera]
que         [nunca, pensé, pudiera, trabajar]
pudiera            [pensé, que, trabajar, en]
trabajar               [que, pudiera, en, un]
en             [pudiera, trabajar, un, banco]
un                      [trabajar, en, banco]
banco                                [en, un]
Name: contexto, dtype: object

In [ ]:
# --- 1.7 Pares de entrenamiento de todo el corpus de noticias ---
n_tokens = sum(len(o) for o in oraciones_noticias)
filas = []
for ventana in (2, 5):
    n_pares = sum(len(pares_skipgram(o, ventana)) for o in oraciones_noticias)
    filas.append({"ventana": ventana, "pares": n_pares, "tokens": n_tokens,
                  "pares por token": round(n_pares / n_tokens, 2),
                  "máximo teórico (2·ventana)": 2 * ventana})

pd.DataFrame(filas).set_index("ventana")

,pares,tokens,pares por token,máximo teórico (2·ventana)
ventana,,,,
2,2296962,605523,3.79,4
5,5433568,605523,8.97,10


### 1.8 Resultados y discusión

#### ¿Vale la pena quitar stopwords o lematizar antes de entrenar embeddings?

No. En los labs anteriores las quitábamos porque el clasificador solo contaba palabras por documento, pero acá lo que importa es la ventana. Si se borran las stopwords, palabras que estaban a cinco posiciones quedan pegadas y el modelo aprende vecinos que en el texto nunca lo fueron. Además ellas mismas aportan, el "la" antes de un sustantivo es lo que le enseña al modelo que es femenino.

Con el stemming es parecido, "gobiernos" y "gobernó" quedan en `gobern` y se pierden el número y el tiempo verbal, que después se necesitan para las analogías.

Al final dejamos minúsculas, sin puntuación, con tildes y con números. La puntuación aquí sí se quita (en el Lab 5 no) porque una coma solo ocuparía un lugar de la ventana. Las noticias además traían párrafos pegados tipo "imaginamos.Para" (6,147 casos en 844 documentos), "EE.UU." partiendo oraciones y números con punto de miles, y hubo que arreglar las tres cosas.

#### ¿Distingue SUC mayúsculas y tildes?

Mayúsculas no, ninguna de las 200,000 palabras que cargamos tiene mayúscula, entonces "España" ni existe y hay que pasar todo a minúsculas sí o sí. Las tildes sí cuentan. "españa" está en la posición 541 y "espana" en la 67,448, así que si quitáramos las tildes las consultas caerían en vectores de palabras rarísimas. Tampoco usa punto de miles ("10000" sí está, "10.000" no).

#### Tamaño de los corpus

| Corpus | Documentos | Oraciones | Tokens | Tipos |
|---|---|---|---|---|
| Noticias | 1,142 | 20,909 | 605,523 | 30,750 |
| Quijote | 126 capítulos | 10,585 | 378,666 | 22,842 |

Con `min_count = 2` el vocabulario de noticias baja de 30,750 a 16,832 y solo se pierde el 2.3% de los tokens, y con 5 queda en 8,595 perdiendo el 5.9% (en el Quijote pasa lo mismo, un poco más fuerte). Casi la mitad de las palabras aparece una sola vez y de ahí no sale un vector que sirva.

#### ¿Por qué word2vec no necesita datos etiquetados?

Con ventana 2 salen **2,296,962** pares y con ventana 5 **5,433,568**, casi 4 y 9 pares por cada token. Cada par ya es un ejemplo de entrenamiento y la "etiqueta" es la palabra de al lado, que viene en el mismo texto. No hace falta que nadie anote nada, por eso SUC se pudo entrenar con 2,600 millones de palabras.

## 2. Co-ocurrencia, PPMI y SVD

Antes de entrenar cualquier red, aplicamos la hipótesis distribucional de la forma más directa, contando qué palabras aparecen cerca de cuáles. Esta sección usa solo el corpus de noticias.

In [ ]:
# --- 2.1 Matriz de co-ocurrencia ±2 sobre las 2,000 palabras más frecuentes ---
from numpy.linalg import norm

conteo_noticias = Counter(t for o in oraciones_noticias for t in o)
vocab_cooc = [w for w, _ in conteo_noticias.most_common(2000)]
idx_cooc = {w: i for i, w in enumerate(vocab_cooc)}

X_cooc = np.zeros((len(vocab_cooc), len(vocab_cooc)), dtype=np.float64)
for o in oraciones_noticias:
    for central, contexto in pares_skipgram(o, ventana=2):
        if central in idx_cooc and contexto in idx_cooc:
            X_cooc[idx_cooc[central], idx_cooc[contexto]] += 1

ceros = (X_cooc == 0).mean()
print(f"Forma: {X_cooc.shape} · ceros: {ceros:.2%} · simétrica: {np.allclose(X_cooc, X_cooc.T)}")
print("BoW del Lab #2 (documentos × vocabulario completo): 98.71 % ceros")

Forma: (2000, 2000) · ceros: 93.78% · simétrica: True
BoW del Lab #2 (documentos × vocabulario completo): 98.71 % ceros


In [ ]:
# --- 2.2 Coseno entre filas de conteos ---
def coseno(u, v):
    return float(u @ v / (norm(u) * norm(v)))


def fila(M, w):
    return M[idx_cooc[w]]


pares_prueba = [("empresa", "compañía"), ("gobierno", "ministerio"), ("banco", "financiera"),
                ("empresa", "año"), ("tecnología", "presidente")]

pd.DataFrame([{"par": f"{a} / {b}", "coseno conteos": round(coseno(fila(X_cooc, a), fila(X_cooc, b)), 3)}
              for a, b in pares_prueba]).set_index("par")

,coseno conteos
par,
empresa / compañía,0.962
gobierno / ministerio,0.783
banco / financiera,0.362
empresa / año,0.445
tecnología / presidente,0.671


In [ ]:
# --- 2.3 Las 10 palabras con más co-ocurrencias ---
def top_fila(M, w, k=10):
    v = fila(M, w)
    orden = np.argsort(-v)
    return [f"{vocab_cooc[j]} ({v[j]:.0f})" if M is X_cooc else f"{vocab_cooc[j]} ({v[j]:.2f})"
            for j in orden if vocab_cooc[j] != w][:k]


pd.DataFrame({w: top_fila(X_cooc, w) for w in ["empresa", "gobierno"]})

,empresa,gobierno
0,la (192),el (214)
1,de (142),del (106)
2,una (62),de (83)
3,que (47),que (62)
4,en (42),y (56)
5,y (32),nacional (36)
6,con (27),en (29)
7,a (25),para (24)
8,para (17),al (24)
9,como (16),por (23)


In [ ]:
# --- 2.4 PPMI con NumPy ---
# PPMI(w, c) = max(0, log2[P(w, c) / (P(w) · P(c))]); los conteos cero dan log(0) = -inf → 0.
def ppmi(X):
    total = X.sum()
    p_wc = X / total
    p_w = X.sum(axis=1, keepdims=True) / total
    p_c = X.sum(axis=0, keepdims=True) / total
    with np.errstate(divide="ignore"):
        pmi = np.log2(p_wc / (p_w * p_c))
    return np.maximum(pmi, 0)


X_ppmi = ppmi(X_cooc)
print(f"PPMI: {X_ppmi.shape} · ceros: {(X_ppmi == 0).mean():.2%} · máximo: {X_ppmi.max():.2f}")

PPMI: (2000, 2000) · ceros: 94.98% · máximo: 12.47


In [ ]:
# --- 2.5 Las 10 palabras más asociadas, ahora con PPMI ---
pd.DataFrame({w: top_fila(X_ppmi, w) for w in ["empresa", "gobierno"]})

,empresa,gobierno
0,telecomunicaciones (5.51),corporativo (7.23)
1,llega (5.22),federal (5.57)
2,especializada (5.12),sánchez (5.39)
3,líder (4.96),nacional (5.26)
4,cualquier (4.71),iván (5.24)
5,estadounidense (4.70),funcionarios (5.17)
6,colombiana (4.59),petro (4.76)
7,criterios (4.59),estará (4.55)
8,española (4.50),llamado (4.52)
9,somos (4.28),duque (4.40)


In [ ]:
# --- 2.5 (cont.) Los mismos pares de 2.2 con PPMI ---
pd.DataFrame([{"par": f"{a} / {b}",
               "conteos": round(coseno(fila(X_cooc, a), fila(X_cooc, b)), 3),
               "PPMI": round(coseno(fila(X_ppmi, a), fila(X_ppmi, b)), 3)}
              for a, b in pares_prueba]).set_index("par")

,conteos,PPMI
par,,
empresa / compañía,0.962,0.264
gobierno / ministerio,0.783,0.098
banco / financiera,0.362,0.131
empresa / año,0.445,0.050
tecnología / presidente,0.671,0.066


In [ ]:
# --- 2.6 SVD a 100 dimensiones: primer modelo de embeddings ---
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
vectores_svd = svd.fit_transform(X_ppmi)

kv_svd = KeyedVectors(vector_size=100)
kv_svd.add_vectors(vocab_cooc, vectores_svd)

print(f"Varianza explicada por 100 componentes: {svd.explained_variance_ratio_.sum():.1%}")
for w in ["empresa", "gobierno", "banco"]:
    print(f"{w:>9} →", [v for v, _ in kv_svd.most_similar(w, topn=5)])

Varianza explicada por 100 componentes: 25.0%
  empresa → ['compañía', 'plataforma', 'entidad', 'tecnología', 'asociación']
 gobierno → ['congreso', 'banco', 'república', 'central', 'también']
    banco → ['grupo', 'bbva', 'central', 'colombia', 'gobierno']


### 2.7 Resultados y discusión

La matriz queda de 2,000 × 2,000 con **93.78%** de ceros, menos que la BoW del Lab 2 (98.71%), pero es porque solo dejamos las 2,000 palabras más frecuentes. Con todo el vocabulario estaría mucho más vacía.

#### ¿Coincide el coseno con su intuición?

Con los conteos no mucho. "empresa / compañía" sale 0.962, bien, pero "tecnología / presidente" da 0.671 y "banco / financiera" apenas 0.362, cuando debería ser al revés. Todos los sustantivos se parecen entre sí porque comparten los mismos vecinos, "el", "la", "de".

#### ¿Qué tipo de palabras dominan la lista?

Stopwords, casi solo stopwords. Para "empresa" el top 10 es la, de, una, que, en, y, con, a, para, como, y "gobierno" es igual con un "nacional" metido. Lo único que eso dice de "empresa" es que es un sustantivo femenino.

#### ¿Qué cambió con PPMI y por qué?

| Par | Conteos | PPMI |
|---|---|---|
| empresa / compañía | 0.962 | 0.264 |
| gobierno / ministerio | 0.783 | 0.098 |
| banco / financiera | 0.362 | 0.131 |
| empresa / año | 0.445 | 0.050 |
| tecnología / presidente | 0.671 | 0.066 |

PPMI compara cuántas veces salen juntas dos palabras contra cuántas se esperaría por azar. Como "la" sale con todo, su PMI con "empresa" es bajo y se va de la lista, y quedan cosas como telecomunicaciones, especializada, líder o colombiana. Para "gobierno" quedan federal, funcionarios, sánchez, petro y duque, que son los gobiernos de los que hablan las noticias.
Los cosenos bajan todos pero el orden ya tiene más sentido, los dos pares sin relación quedan abajo. Sigue habiendo ruido ("llega", "cualquier") porque PMI exagera con los pares raros.

#### ¿Por qué las filas ya son vectores de palabras y qué resuelven PPMI y SVD?

Cada fila describe una palabra por los contextos donde aparece, y si dos palabras se usan parecido sus filas se parecen. El problema es que son 2,000 dimensiones casi todas en cero y dominadas por las stopwords. PPMI le quita el peso a las stopwords y SVD comprime todo a 100 dimensiones densas donde los contextos parecidos quedan en la misma dirección. Con eso los vecinos de "empresa" ya son compañía, plataforma, entidad, tecnología y asociación.

## 3. Investigación

#### ¿En qué se diferencian CBOW y skip-gram? ¿Cuál se espera que funcione mejor con palabras poco frecuentes?

CBOW promedia los vectores de las palabras del contexto y con eso predice la palabra central, mientras que skip-gram hace lo contrario y usa la palabra central para predecir cada palabra del contexto. CBOW es más rápido, pero al promediar se diluyen las palabras raras.
En skip-gram cada vez que aparece una palabra rara se generan sus propios pares de entrenamiento, por lo tanto funciona mejor con palabras poco frecuentes.

#### ¿Qué problema resuelve negative sampling frente a un softmax sobre todo el vocabulario?

Con softmax, cada predicción tiene que normalizar sobre todo el vocabulario (16,832 palabras en nuestro caso y millones en SUC), y eso se paga en cada uno de los pares. Negative sampling cambia ese problema por varias clasificaciones binarias, donde se compara el par real contra *k* palabras sorteadas al azar.

P(real | w, c) = σ(v<sub>w</sub> · c<sub>c</sub>) = 1 / (1 + e<sup>−v<sub>w</sub> · c<sub>c</sub></sup>)

Aquí *v<sub>w</sub>* es el vector de la palabra central en la tabla de palabras y *c<sub>c</sub>* es el vector de la palabra de contexto en la tabla de contextos. Si el producto punto es alto, la probabilidad se acerca a 1 y el modelo cree que el par sí aparece en el texto.

#### ¿Por qué word2vec aprende dos tablas de vectores y cuál se conserva al final?

Porque una palabra casi nunca es su propio contexto. Con una sola tabla, el producto de una palabra consigo misma siempre sería alto y el modelo creería que cada palabra aparece al lado de sí misma, cosa que casi nunca pasa. Al final se conserva la tabla de palabras, que en gensim es `model.wv`.

#### ¿En qué se diferencian word2vec, GloVe y fastText? ¿Por qué fastText es especialmente útil en español?

Word2vec es predictivo y local, ya que aprende ventana por ventana. GloVe en cambio parte de la matriz global de co-ocurrencias, parecido al SVD de la sección 2, pero la factoriza con un objetivo de regresión ponderada.
FastText representa cada palabra como la suma de sus n-gramas de caracteres, y por eso puede armar un vector para palabras que nunca vio. En español eso ayuda mucho porque un solo verbo tiene decenas de conjugaciones y muchas de ellas no aparecen en el corpus.

#### Parámetros de `gensim.models.Word2Vec`

| Parámetro | Propósito |
|---|---|
| `vector_size` | Dimensión de los vectores |
| `window` | Cuántas palabras de contexto se toman a cada lado de la central |
| `min_count` | Frecuencia mínima para entrar al vocabulario |
| `sg` | 1 = skip-gram, 0 = CBOW |
| `negative` | Cuántas palabras negativas se sortean por cada par |
| `sample` | Umbral para descartar al azar las palabras muy frecuentes |
| `epochs` | Cuántas pasadas se hacen sobre el corpus |

#### `nn.Embedding`, `from_pretrained` y `nn.EmbeddingBag`

`nn.Embedding(num_embeddings, embedding_dim, padding_idx)` es una tabla entrenable con `num_embeddings` filas y `embedding_dim` columnas, y la fila `padding_idx` se queda en cero y no recibe gradiente. `nn.Embedding.from_pretrained(E, freeze=True)` la inicializa con vectores que ya fueron entrenados, y con `freeze=False` esos vectores también se ajustan durante el entrenamiento (fine-tuning).
`nn.EmbeddingBag` busca los vectores de una secuencia y los combina en uno solo en el mismo paso según `mode` (`"mean"`, `"sum"` o `"max"`), sin tener que armar la matriz intermedia.

Multiplicar un vector one-hot por la matriz E equivale a seleccionar una fila porque el one-hot tiene un único 1 y todo lo demás es cero. El producto solo deja la fila de E que está en esa posición, por eso `nn.Embedding` indexa directamente en vez de hacer la multiplicación.

## 4. Entrenar Word2Vec y fastText

Entrenamos nuestros propios embeddings con gensim sobre las oraciones de la sección 1.

In [ ]:
# --- 4.1 Modelo base ---
import contextlib
import io
import zlib

from gensim.models import Word2Vec, FastText


# Reproducible: un solo hilo y un hash que no cambie entre sesiones (el hash() de Python sí cambia).
def hash_estable(texto):
    return zlib.crc32(texto.encode("utf-8"))


BASE = dict(vector_size=100, window=5, min_count=2, sg=1, negative=5, epochs=10,
            workers=1, seed=RANDOM_STATE, hashfxn=hash_estable)


def entrena(oraciones, clase=Word2Vec, **cambios):
    t0 = time.time()
    # gensim 4.4 con NumPy 2.5 imprime un "Exception ignored" inofensivo en cada época.
    with contextlib.redirect_stderr(io.StringIO()):
        modelo = clase(sentences=oraciones, **{**BASE, **cambios})
    return modelo, time.time() - t0


w2v_base, t_base = entrena(oraciones_noticias)
print(f"Base: {len(w2v_base.wv):,} palabras × {w2v_base.vector_size} dims en {t_base:.1f} s")

Base: 16,832 palabras × 100 dims en 10.2 s


In [ ]:
# --- 4.2 Variantes, un parámetro a la vez ---
VARIANTES = {
    "base (sg, w5, d100)": {},
    "CBOW": dict(sg=0),
    "ventana 2": dict(window=2),
    "ventana 10": dict(window=10),
    "dim 50": dict(vector_size=50),
    "dim 300": dict(vector_size=300),
}

modelos, tiempos = {"base (sg, w5, d100)": w2v_base}, {"base (sg, w5, d100)": t_base}
for nombre, cambios in VARIANTES.items():
    if nombre not in modelos:
        modelos[nombre], tiempos[nombre] = entrena(oraciones_noticias, **cambios)

pd.DataFrame({"cambio": [str(c) if c else "—" for c in VARIANTES.values()],
              "segundos": [round(tiempos[n], 1) for n in VARIANTES]}, index=list(VARIANTES))

,cambio,segundos
"base (sg, w5, d100)",—,10.2
CBOW,{'sg': 0},3.9
ventana 2,{'window': 2},6.6
ventana 10,{'window': 10},15.6
dim 50,{'vector_size': 50},8.0
dim 300,{'vector_size': 300},19.2


In [ ]:
# --- 4.3 Palabras de control ---
CONTROL = ["banco", "gobierno", "empresa", "mercado", "tecnología", "sostenibilidad"]
print({w: conteo_noticias[w] for w in CONTROL})
assert all(w in w2v_base.wv for w in CONTROL)

{'banco': 1097, 'gobierno': 367, 'empresa': 323, 'mercado': 737, 'tecnología': 349, 'sostenibilidad': 363}


In [ ]:
# --- 4.4 Vecinos de cada palabra de control por variante ---
def vecinos(kv, w, k=5):
    return ", ".join(v for v, _ in kv.most_similar(w, topn=k)) if w in kv else "(fuera de vocab.)"


tabla_variantes = pd.DataFrame({n: [vecinos(m.wv, w) for w in CONTROL] for n, m in modelos.items()},
                               index=CONTROL).T
with pd.option_context("display.max_colwidth", None):
    display(tabla_variantes)

,banco,gobierno,empresa,mercado,tecnología,sostenibilidad
"base (sg, w5, d100)","central, popular, agrario, sudameris, república","congreso, plan, estímulo, mandatario, biden","startup, liga, veridas, plataforma, certificación","sector, colombiano, solvencia, apetito, liquidez","computación, personalización, estandarización, innovación, cuántica","digitalización, estratégicas, diversidad, género, creatividad"
CBOW,"cordillera, grupo, equipo, mercado, área","congreso, estado, tema, paquete, clima","plataforma, compañía, marca, startup, aplicación","país, planeta, mundo, sector, territorio","digitalización, innovación, sociedad, información, transformación","innovación, digitalización, transformación, empresarial, estrategia"
ventana 2,"popular, bce, central, gnb, scotiabank","congreso, bce, gremial, consejo, senado","startup, compañía, plataforma, convocatoria, película","sector, territorio, escenario, entorno, rezago","ciencia, computación, publicidad, educativa, funcionalidad","digitalización, sociedad, diversidad, innovación, emprendimiento"
ventana 10,"central, popular, república, agrario, occidente","congreso, biden, asumir, ministerio, detalló","startup, compañía, plataforma, conjuntamente, multinacional","sector, accionario, colombiano, jugador, koin","estandarización, innovadora, nube, punteras, tecnológica","prioridades, alineado, sociedad, suscríbete, estratégicas"
dim 50,"central, popular, agrario, bancos, entidad","congreso, duque, tono, plan, sec","startup, plataforma, enisa, emergente, monitoreo","creciente, sector, colombiano, apetito, sólido","capacitación, personalización, estandarización, gamificación, computación","digitalización, sociedad, innovación, misión, diversidad"
dim 300,"central, popular, agrario, república, gnb","congreso, plan, estímulo, senado, biden","startup, liga, emergente, innovadora, veridas","colombiano, sector, mercados, solvencia, accionario","computación, estandarización, personalización, cuántica, innovación","digitalización, estratégicas, diversidad, internacionalización, sociedad"


In [ ]:
# --- 4.5 El mismo modelo base sobre el Quijote ---
w2v_quijote, t_q = entrena(oraciones_quijote)
print(f"Quijote: {len(w2v_quijote.wv):,} palabras en {t_q:.1f} s\n")

COMUNES = ["señor", "dinero", "casa", "caballo", "palabra"]
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({"noticias": [vecinos(w2v_base.wv, w) for w in COMUNES],
                          "Quijote": [vecinos(w2v_quijote.wv, w) for w in COMUNES]}, index=COMUNES))

Quijote: 11,690 palabras en 6.4 s



,noticias,Quijote
señor,"coincidimos, brazo, comunique, colapsar, parafraseando","álvaro, tarfe, castellano, perdóneme, alonso"
dinero,"gastar, vivir, permitía, crediticio, recibir","hallazgo, pareciéndome, capellán, dueño, sacristán"
casa,"pasiva, filtro, enfoca, teletrabajo, terraza","aldea, compañía, mujer, félix, hacienda"
caballo,(fuera de vocab.),"freno, madera, rodado, carro, pie"
palabra,"tantos, agravado, corrupción, inevitable, especialidad","hablar, hablase, respondelle, suspirar, responder"


In [ ]:
# --- 4.6 FastText y palabras fuera de vocabulario ---
ft_noticias, t_ft = entrena(oraciones_noticias, clase=FastText)
print(f"FastText: {t_ft:.1f} s\n")

OOV = {"error de dedo": "goberino", "conjugación rara": "invirtiéramos", "inventada": "criptoempresa"}
filas = []
for tipo, w in OOV.items():
    try:
        w2v = vecinos(w2v_base.wv, w) if w in w2v_base.wv else str(w2v_base.wv[w])
    except KeyError as e:
        w2v = f"KeyError: {e}"
    filas.append({"tipo": tipo, "palabra": w, "¿en vocab. FastText?": w in ft_noticias.wv.key_to_index,
                  "FastText": vecinos(ft_noticias.wv, w), "Word2Vec": w2v})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(filas).set_index("tipo"))

FastText: 24.7 s



,palabra,¿en vocab. FastText?,FastText,Word2Vec
tipo,,,,
error de dedo,goberino,False,"riders, declaren, gobernanza, guardián, onyze","KeyError: ""Key 'goberino' not present"""
conjugación rara,invirtiéramos,False,"invitamos, aspiramos, imaginamos, miramos, acabamos","KeyError: ""Key 'invirtiéramos' not present"""
inventada,criptoempresa,False,"empresa, criptografía, expresa, criptomoneda, cripto","KeyError: ""Key 'criptoempresa' not present"""


In [ ]:
# --- 4.7 Mejor modelo propio ---
MEJOR = "base (sg, w5, d100)"
w2v_mejor = modelos[MEJOR]
w2v_mejor.wv.save("w2v_noticias.kv")
print(f"Mejor modelo: {MEJOR} → w2v_noticias.kv")

Mejor modelo: base (sg, w5, d100) → w2v_noticias.kv


### 4.8 Resultados y discusión

Los modelos tardan entre 3 y 19 segundos. Los corrimos con un solo hilo porque con varios cada corrida daba vecinos un poco distintos y no se podía comparar nada.

CBOW es el más rápido y el peor. Para "banco" da cordillera, grupo, equipo, mercado y área, nada que ver. Promediar el contexto le quita lo específico a cada palabra y con un corpus tan chico se nota.

#### ¿Las ventanas pequeñas dan vecinos más sintácticos y las grandes, más temáticos?

Más o menos. Con ventana 2, "banco" da popular, bce, central, gnb y scotiabank, y "gobierno" da congreso, bce, gremial, consejo y senado, palabras que podrían ir en el mismo lugar de la oración. Con ventana 10 ya aparecen palabras del mismo tema que no se pueden intercambiar, "gobierno" trae biden, asumir, ministerio y detalló, y "sostenibilidad" trae prioridades, alineado, estratégicas y hasta "suscríbete", que sale del pie de las notas.
Lo que casi no vimos son otras formas de la misma palabra (solo "bancos" con 50 dimensiones y "mercados" con 300), seguramente porque no hay tantos ejemplos de cada forma.

#### ¿Qué dice esto sobre la frase "el significado viene del corpus"?

| Palabra | Noticias | Quijote |
|---|---|---|
| señor | coincidimos, brazo, comunique, colapsar | álvaro, tarfe, castellano, perdóneme |
| casa | pasiva, filtro, enfoca, teletrabajo | aldea, compañía, mujer, hacienda |
| caballo | (fuera de vocabulario) | freno, madera, rodado, carro |
| palabra | tantos, agravado, corrupción | hablar, hablase, respondelle, responder |

En las noticias "señor" y "casa" casi no salen, por lo tanto sus vecinos son ruido, y "caballo" ni llega a `min_count = 2`. En el Quijote "señor" es un tratamiento (don Álvaro Tarfe, perdóneme), "casa" queda con aldea y hacienda y "caballo" con freno y carro. El vector no guarda lo que la palabra significa en general sino cómo se usa en ese texto.

#### FastText y palabras fuera de vocabulario

Word2Vec tira `KeyError` con las tres palabras porque si la palabra no está en la tabla no tiene vector. FastText sí arma uno con los n-gramas de caracteres. "criptoempresa" sale cerca de empresa, criptografía y criptomoneda, que tiene todo el sentido. "invirtiéramos" sale con invitamos, aspiramos e imaginamos, o sea que agarra la forma pero no el verbo. Con "goberino" le fue mal, fuera de "gobernanza" son palabras que solo comparten pedazos (riders, declaren, guardián).

#### ¿Cuál es el mejor modelo?

El base, skip-gram con ventana 5 y 100 dimensiones. Da vecinos igual de buenos que el de 300 (para banco salen central, popular, agrario, sudameris y república) en la mitad del tiempo, y tiene las mismas dimensiones que SUC, entonces la comparación de la sección 5 es más justa. Quedó guardado en `w2v_noticias.kv`.

## 5. Similitud, vecinos y analogías

Comparamos cuatro modelos, el SVD sobre PPMI de la sección 2, nuestro mejor Word2Vec de noticias, el Word2Vec del Quijote y fastText SUC.

In [ ]:
# --- 5.1 Coseno propio contra gensim ---
MODELOS = {"SVD-PPMI": kv_svd, "W2V noticias": w2v_mejor.wv,
           "W2V Quijote": w2v_quijote.wv, "fastText SUC": suc}

kv = w2v_mejor.wv
pd.DataFrame([{"par": f"{a} / {b}", "coseno propio": coseno(kv[a], kv[b]),
               "gensim similarity": float(kv.similarity(a, b))}
              for a, b in [("empresa", "compañía"), ("banco", "dinero"), ("gobierno", "mercado")]]
             ).set_index("par")

,coseno propio,gensim similarity
par,,
empresa / compañía,0.612982,0.612982
banco / dinero,0.158376,0.158376
gobierno / mercado,0.217781,0.217781


In [ ]:
# --- 5.1 (cont.) Cinco vecinos en los cuatro modelos ---
PALABRAS = ["banco", "gobierno", "dinero", "precio", "mujer"]
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({n: [vecinos(m, w) for w in PALABRAS] for n, m in MODELOS.items()},
                         index=PALABRAS))

,SVD-PPMI,W2V noticias,W2V Quijote,fastText SUC
banco,"grupo, bbva, central, colombia, gobierno","central, popular, agrario, sudameris, república","ristre, escarlata, marfil, desenvainó, corrediza","hipotecario, bancos, bank, citibank, monetario"
gobierno,"congreso, banco, república, central, también","congreso, plan, estímulo, mandatario, biden","condado, pago, habré, pulso, prometido","elgobierno, gobier, go­bierno, ministro, gobiernos"
dinero,"liquidez, hacer, manejo, momento, no","gastar, vivir, permitía, crediticio, recibir","hallazgo, pareciéndome, capellán, dueño, sacristán","dinerito, dineral, centavo, gastarlo, dinerillo"
precio,"costo, precios, tasa, gasto, mensual","gasolina, costos, frescos, precios, litro","hubiéredes, partí, llevamos, durante, pliegos","coste, precios, costo, predo, reventa"
mujer,(fuera de vocab.),"remodelando, articuladora, dibujan, taxonomía, refuerce","teresa, marido, honrada, hija, madre","niña, muchacha, prostituta, chica, dama"


In [ ]:
# --- 5.1 (cont.) Antónimos ---
ANTONIMOS = [("bueno", "malo"), ("subir", "bajar"), ("ganar", "perder"),
             ("público", "privado"), ("alto", "bajo"), ("comprar", "vender")]


def sim(kv, a, b):
    return round(float(kv.similarity(a, b)), 3) if a in kv and b in kv else np.nan


pd.DataFrame({n: [sim(m, a, b) for a, b in ANTONIMOS] for n, m in MODELOS.items()},
             index=[f"{a} / {b}" for a, b in ANTONIMOS])

,SVD-PPMI,W2V noticias,W2V Quijote,fastText SUC
bueno / malo,NaN,0.796,0.806,0.717
subir / bajar,NaN,0.773,0.805,0.937
ganar / perder,NaN,0.725,0.706,0.689
público / privado,0.592,0.734,NaN,0.653
alto / bajo,0.355,0.460,0.727,0.514
comprar / vender,NaN,0.808,0.793,0.880


In [ ]:
# --- 5.2 Conjunto de similitud propio ---
# Puntajes 0–10 a mano, en escala de relación temática (estilo WordSim-353), no de sinonimia.
from scipy.stats import spearmanr

GOLD = [("empresa", "compañía", 9.5), ("dinero", "plata", 9.0), ("presidente", "mandatario", 9.0),
        ("empleo", "trabajo", 9.0), ("casa", "hogar", 9.0), ("precio", "costo", 9.0),
        ("gobierno", "estado", 8.5), ("crisis", "recesión", 8.5), ("banco", "entidad", 7.0),
        ("mercado", "bolsa", 8.0), ("tecnología", "innovación", 8.0), ("inflación", "precios", 8.5),
        ("banco", "dinero", 7.5), ("hombre", "mujer", 7.0), ("ciudad", "país", 6.0),
        ("caballo", "perro", 4.0), ("libro", "fútbol", 1.0), ("agua", "impuesto", 1.5),
        ("señor", "tecnología", 0.5), ("sol", "empresa", 0.5)]

filas = []
for nombre, m in MODELOS.items():
    dentro = [(a, b, s) for a, b, s in GOLD if a in m and b in m]
    rho, p = spearmanr([s for _, _, s in dentro], [m.similarity(a, b) for a, b, _ in dentro])
    filas.append({"modelo": nombre, "pares evaluados": len(dentro),
                  "fuera de vocab.": len(GOLD) - len(dentro), "Spearman ρ": round(rho, 3),
                  "p-valor": round(p, 4)})
pd.DataFrame(filas).set_index("modelo")

,pares evaluados,fuera de vocab.,Spearman ρ,p-valor
modelo,,,,
SVD-PPMI,14,6,0.422,0.1332
W2V noticias,19,1,0.342,0.1516
W2V Quijote,7,13,0.378,0.4026
fastText SUC,20,0,0.607,0.0045


In [ ]:
# --- 5.2 (cont.) Detalle por par ---
pd.DataFrame({"mi puntaje": [s for _, _, s in GOLD],
              **{n: [sim(m, a, b) for a, b, _ in GOLD] for n, m in MODELOS.items()}},
             index=[f"{a} / {b}" for a, b, _ in GOLD])

,mi puntaje,SVD-PPMI,W2V noticias,W2V Quijote,fastText SUC
empresa / compañía,9.5,0.730,0.613,0.664,0.824
dinero / plata,9.0,NaN,0.452,0.348,0.447
presidente / mandatario,9.0,NaN,0.658,NaN,0.691
empleo / trabajo,9.0,0.325,0.458,NaN,0.700
casa / hogar,8.5,0.347,0.280,NaN,0.608
precio / costo,8.5,0.640,0.655,NaN,0.727
gobierno / estado,7.5,0.484,0.597,0.670,0.565
crisis / recesión,7.5,0.444,0.570,NaN,0.809
banco / entidad,7.0,0.521,0.448,NaN,0.419
mercado / bolsa,7.0,0.305,0.197,NaN,0.415


In [ ]:
# --- 5.3 Analogías, a es a b como c es a ? ---
def analogia(kv, a, b, c, k=5, excluir=True):
    """Vector b − a + c sobre vectores normalizados; devuelve las k más cercanas."""
    V = kv.get_normed_vectors()
    u = lambda w: V[kv.key_to_index[w]]
    objetivo = u(b) - u(a) + u(c)
    sims = V @ (objetivo / norm(objetivo))
    orden = np.argsort(-sims)
    fuera = {a, b, c} if excluir else set()
    return [(kv.index_to_key[i], float(sims[i])) for i in orden[:k + 3]
            if kv.index_to_key[i] not in fuera][:k]


for a, b, c in [("hombre", "rey", "mujer"), ("españa", "madrid", "francia")]:
    propio = [w for w, _ in analogia(suc, a, b, c)]
    de_gensim = [w for w, _ in suc.most_similar(positive=[b, c], negative=[a], topn=5)]
    print(f"{a} : {b} :: {c} : ?")
    print(f"   propio {propio}")
    print(f"   gensim {de_gensim}  iguales: {propio == de_gensim}")

hombre : rey :: mujer : ?
   propio ['reina', 'princesa', 'consorte', 'sibila', 'isabel']
   gensim ['reina', 'princesa', 'consorte', 'sibila', 'isabel']  iguales: True
españa : madrid :: francia : ?
   propio ['toulouse', 'parís', 'lille', 'montpellier', 'rennes']
   gensim ['toulouse', 'parís', 'lille', 'montpellier', 'rennes']  iguales: True


In [ ]:
# --- 5.3 (cont.) Analogías propias, seis tipos ---
ANALOGIAS = [
    ("género", "hombre", "rey", "mujer", "reina"),
    ("género", "hombre", "padre", "mujer", "madre"),
    ("género", "hombre", "hijo", "mujer", "hija"),
    ("país–capital", "francia", "parís", "italia", "roma"),
    ("país–capital", "españa", "madrid", "colombia", "bogotá"),
    ("país–capital", "alemania", "berlín", "japón", "tokio"),
    ("país–gentilicio", "colombia", "colombiano", "méxico", "mexicano"),
    ("país–gentilicio", "españa", "español", "francia", "francés"),
    ("país–gentilicio", "chile", "chileno", "argentina", "argentino"),
    ("singular–plural", "empresa", "empresas", "banco", "bancos"),
    ("singular–plural", "mercado", "mercados", "país", "países"),
    ("singular–plural", "precio", "precios", "año", "años"),
    ("tiempo verbal", "comía", "comerá", "vivía", "vivirá"),
    ("tiempo verbal", "compra", "compró", "vende", "vendió"),
    ("tiempo verbal", "crece", "creció", "cae", "cayó"),
    ("adjetivo m–f", "nuevo", "nueva", "alto", "alta"),
    ("adjetivo m–f", "bueno", "buena", "pequeño", "pequeña"),
    ("adjetivo m–f", "colombiano", "colombiana", "mexicano", "mexicana"),
]


def posicion(kv, a, b, c, d, n=100):
    if not all(w in kv for w in (a, b, c, d)):
        return None, "OOV"
    palabras = [w for w, _ in analogia(kv, a, b, c, k=n)]
    return palabras[:5], palabras.index(d) + 1 if d in palabras else f">{n}"


resultados = []
for tipo, a, b, c, d in ANALOGIAS:
    for nombre, m in MODELOS.items():
        top5, pos = posicion(m, a, b, c, d)
        resultados.append({"tipo": tipo, "analogía": f"{a}:{b} :: {c}:{d}", "modelo": nombre,
                           "top 5": ", ".join(top5) if top5 else "—", "posición": pos})
res_analogias = pd.DataFrame(resultados)

# Posición de la respuesta correcta en cada modelo (el top 5 completo está en res_analogias).
res_analogias.pivot(index="analogía", columns="modelo", values="posición").reindex(
    [f"{a}:{b} :: {c}:{d}" for _, a, b, c, d in ANALOGIAS])

modelo,SVD-PPMI,W2V Quijote,W2V noticias,fastText SUC
analogía,,,,
hombre:rey :: mujer:reina,OOV,27,>100,1
hombre:padre :: mujer:madre,OOV,5,>100,1
hombre:hijo :: mujer:hija,OOV,11,>100,1
francia:parís :: italia:roma,OOV,>100,>100,2
españa:madrid :: colombia:bogotá,54,OOV,>100,1
alemania:berlín :: japón:tokio,OOV,OOV,>100,1
colombia:colombiano :: méxico:mexicano,OOV,OOV,1,1
españa:español :: francia:francés,OOV,>100,>100,1
chile:chileno :: argentina:argentino,OOV,OOV,>100,1


In [ ]:
# --- 5.3 (cont.) Top 5 de cada analogía y modelo ---
with pd.option_context("display.max_colwidth", None, "display.max_rows", 100):
    display(res_analogias.set_index(["analogía", "modelo"])[["top 5", "posición"]])

top 5  \
analogía                                   modelo                                                                         
hombre:rey :: mujer:reina                  SVD-PPMI                                                                   —   
                                           W2V noticias                   rincón, coordinador, castellanos, ana, iberia   
                                           W2V Quijote                       madre, patria, félix, antonomasia, infanta   
                                           fastText SUC                       reina, princesa, consorte, sibila, isabel   
hombre:padre :: mujer:madre                SVD-PPMI                                                                   —   
                                           W2V noticias  liderar, concienciar, representación, científica, desarrollará   
                                           W2V Quijote                                 marido, hijos, félix, ana, madre   
                                           fastText SUC                           madre, hija, hermana, esposa, sobrina   
hombre:hijo :: mujer:hija                  SVD-PPMI                                                                   —   
                                           W2V noticias               implementado, inmersivo, acierto, planteado, alba   
                                           W2V Quijote                           madre, teresa, marido, legítima, hijos   
                                           fastText SUC                             hija, esposa, nieta, nuera, sobrina   
francia:parís :: italia:roma               SVD-PPMI                                                                   —   
                                           W2V noticias     alineamiento, consecución, fijado, climáticamente, fijación   
                                           W2V Quijote                     tormentas, agarenos, trapa, mejorado, verbis   
                                           fastText SUC                             milán, roma, turín, venecia, verona   
españa:madrid :: colombia:bogotá           SVD-PPMI                              presidente, juan, sede, aseguró, duque   
                                           W2V noticias                        galicia, cartagena, cali, enel, medellín   
                                           W2V Quijote                                                                —   
                                           fastText SUC                  bogotá, ibagué, bucaramanga, medellín, caracas   
alemania:berlín :: japón:tokio             SVD-PPMI                                                                   —   
                                           W2V noticias                 tuiteó, ratings, finlandés, harrisburg, arquero   
                                           W2V Quijote                                                                —   
                                           fastText SUC                           tokio, osaka, tokyo, sapporo, fukuoka   
colombia:colombiano :: méxico:mexicano     SVD-PPMI                                                                   —   
                                           W2V noticias          mexicano, chileno, peso, antiinflacionario, impactarán   
                                           W2V Quijote                                                                —   
                                           fastText SUC          mexicano, michoacano, jalisco, veracruzano, mexiquense   
españa:español :: francia:francés          SVD-PPMI                                                                   —   
                                           W2V noticias                                  ibex, 031, stoxx, roe, galicia   
                                           W2V Quijote                caminado, apartado, cochero, atraviesa, forastero   
                                           fastText SUC

In [ ]:
# --- 5.3 (cont.) Aciertos top 1 y top 5 por tipo (OOV cuenta como fallo) ---
res_analogias["top1"] = res_analogias["posición"].apply(lambda p: isinstance(p, int) and p == 1)
res_analogias["top5"] = res_analogias["posición"].apply(lambda p: isinstance(p, int) and p <= 5)

resumen_analogias = (res_analogias.groupby(["tipo", "modelo"])[["top1", "top5"]].mean().mul(100)
                     .round(0).unstack("modelo"))
totales = res_analogias.groupby("modelo")[["top1", "top5"]].mean().mul(100).round(0).unstack()
resumen_analogias.loc["total"] = totales[resumen_analogias.columns]

print("Analogías con alguna palabra fuera de vocabulario:",
      res_analogias[res_analogias["posición"] == "OOV"].groupby("modelo").size().to_dict())
resumen_analogias

Analogías con alguna palabra fuera de vocabulario: {'SVD-PPMI': 13, 'W2V Quijote': 9, 'W2V noticias': 1}


top1                                           top5  \
modelo          SVD-PPMI W2V Quijote W2V noticias fastText SUC SVD-PPMI   
tipo                                                                      
adjetivo m–f         0.0        33.0          0.0         67.0      0.0   
género               0.0         0.0          0.0        100.0      0.0   
país–capital         0.0         0.0          0.0         67.0      0.0   
país–gentilicio      0.0         0.0         33.0        100.0      0.0   
singular–plural     33.0         0.0          0.0         67.0     33.0   
tiempo verbal        0.0         0.0          0.0         67.0      0.0   
total                6.0         6.0          6.0         78.0      6.0   

                                                       
modelo          W2V Quijote W2V noticias fastText SUC  
tipo                                                   
adjetivo m–f           33.0          0.0        100.0  
género                 33.0          0.0        100.0  
país–capital            0.0          0.0        100.0  
país–gentilicio         0.0         33.0        100.0  
singular–plural         0.0         33.0         67.0  
tiempo verbal           0.0          0.0        100.0  
total                  11.0         11.0         94.0

In [ ]:
# --- 5.3 (cont.) Sin excluir las palabras de la consulta ---
SIN_EXCLUIR = [ANALOGIAS[i] for i in (0, 3, 6, 9, 13)]
pd.DataFrame([{"analogía": f"{a}:{b} :: {c}:{d}", "modelo": n,
               "top 3 sin excluir": ", ".join(w for w, _ in analogia(MODELOS[n], a, b, c, k=3, excluir=False))}
              for _, a, b, c, d in SIN_EXCLUIR for n in ["W2V noticias", "fastText SUC"]
              if all(w in MODELOS[n] for w in (a, b, c))])

,analogía,modelo,top 3 sin excluir
0,hombre:rey :: mujer:reina,W2V noticias,"rincón, mujer, coordinador"
1,hombre:rey :: mujer:reina,fastText SUC,"reina, rey, princesa"
2,francia:parís :: italia:roma,W2V noticias,"parís, alineamiento, consecución"
3,francia:parís :: italia:roma,fastText SUC,"milán, roma, turín"
4,colombia:colombiano :: méxico:mexicano,W2V noticias,"colombiano, mexicano, méxico"
5,colombia:colombiano :: méxico:mexicano,fastText SUC,"mexicano, michoacano, jalisco"
6,empresa:empresas :: banco:bancos,W2V noticias,"empresas, banco, central"
7,empresa:empresas :: banco:bancos,fastText SUC,"banco, bancos, crediticias"
8,compra:compró :: vende:vendió,W2V noticias,"vende, pasaba, cuántico"
9,compra:compró :: vende:vendió,fastText SUC,"compró, vendió, vende"


### 5.4 Resultados y discusión

La función `coseno` da lo mismo que gensim en los tres pares (0.613, 0.158 y 0.218) y `analogia` devuelve el mismo top 5 que `most_similar`, así que por ese lado todo bien.

Los vecinos dependen mucho del tamaño de cada modelo. SUC da vecinos limpios para casi todo, por ejemplo para dinero salen dinerito, dineral y centavo. El nuestro está bien con las palabras que las noticias repiten (banco, gobierno, precio) pero con "mujer" da puro ruido porque casi no aparece. Con el Quijote pasa lo contrario, "mujer" sale con teresa, marido, hija y madre, y "banco" con ristre y desenvainó.

#### ¿Son altas o bajas las similitudes entre antónimos?

Altas. Casi todas pasan de 0.6 y "subir / bajar" llega a 0.937 en SUC. Los antónimos aparecen en los mismos contextos (la tasa sube o baja, las acciones se compran o se venden) y el modelo solo ve eso. Por lo tanto la similitud distribucional mide qué tan intercambiables son dos palabras en una oración, y no si significan lo mismo.

#### ¿Qué modelo se parece más a su juicio?

Los puntajes los pusimos en escala de relación, como en WordSim-353, y no de sinonimia. O sea que "inflación / precios" tiene 8.5 aunque no signifiquen lo mismo, que es justo lo que mide un embedding según lo que vimos con los antónimos.

| Modelo | Pares evaluados | Fuera de vocab. | Spearman ρ | p-valor |
|---|---|---|---|---|
| SVD-PPMI | 14 | 6 | 0.499 | 0.069 |
| W2V noticias | 19 | 1 | 0.342 | 0.152 |
| W2V Quijote | 7 | 13 | 0.393 | 0.383 |
| fastText SUC | 20 | 0 | **0.666** | **0.001** |

SUC, y por bastante. Es el único con un p-valor bajo, el SVD se le acerca con 0.499 pero sobre solo 14 pares y los otros dos no se pueden distinguir del azar. El nuestro falla justo en los pares sin relación, "libro / fútbol" le da 0.536 y "señor / tecnología" 0.499, y a "banco / dinero", que para nosotros vale 7.5, le da apenas 0.158.

#### Analogías

SUC acierta 14 de las 18 en primer lugar (78%) y 17 en el top 5 (94%). La única que no encuentra es "precio:precios :: año:años", donde devuelve finales, mediados y comienzos. Nuestros tres modelos se quedan entre 6% y 11%, el de noticias solo acierta "mexicano" y pone "bancos" segundo. Para que la resta entre vectores sea constante hacen falta muchísimos ejemplos de cada relación, y 600 mil tokens no dan.

Sin excluir las palabras de la consulta, en 4 de 5 casos nuestro Word2Vec devuelve primero una de ellas (por ejemplo "parís" para italia), y SUC en 2 de 5. El vector b − a + c casi siempre queda más cerca de b o de c que de la respuesta, por eso hay que excluirlas. El "≈" no es una igualdad, solo dice que reina es lo más cercano que queda después de sacar lo obvio.

## 6. Visualización del espacio

In [ ]:
# --- 6.1 Seis grupos temáticos presentes en los dos modelos ---
GRUPOS = {
    "países": "colombia méxico españa chile perú argentina brasil venezuela ecuador china japón alemania francia italia rusia india canadá uruguay panamá cuba bolivia paraguay corea suiza portugal inglaterra australia israel egipto turquía grecia suecia noruega",
    "meses": "enero febrero marzo abril mayo junio julio agosto septiembre octubre noviembre diciembre",
    "números": "dos tres cuatro cinco seis siete ocho nueve diez once doce quince veinte treinta cuarenta cincuenta cien mil millones cero trece setenta noventa doscientos",
    "movimiento": "ir venir llegar salir entrar subir bajar caminar correr viajar volver regresar mover avanzar cruzar huir partir acercarse alejarse moverse trasladar desplazar llevar",
    "economía": "inflación precio precios tasa tasas interés deuda crédito inversión inversiones mercado bolsa acciones dólar peso economía pib impuesto impuestos banco bancos ahorro liquidez déficit exportaciones capital dividendos utilidades ingresos ganancias pérdidas",
    "profesiones": "médico abogado ingeniero profesor periodista arquitecto contador enfermera economista científico investigador presidente ministro gerente director empresario policía juez analista consultor programador diseñador maestro maestra profesora abogada ingeniera piloto secretaria doctor asesor técnico banquero",
}
palabras_viz, grupo_viz = [], []
for grupo, lista in GRUPOS.items():
    for w in lista.split():
        if w in w2v_mejor.wv and w in suc and w not in palabras_viz:
            palabras_viz.append(w)
            grupo_viz.append(grupo)

print(f"{len(palabras_viz)} palabras:", dict(Counter(grupo_viz)))

In [ ]:
# --- 6.2 PCA y t-SNE en los dos modelos ---
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

COLORES = dict(zip(GRUPOS, plt.cm.tab10.colors))
ROTULOS = {"colombia", "china", "enero", "julio", "diez", "millones", "llegar", "volver",
           "inflación", "dólar", "banco", "médico", "presidente", "periodista"}

fig, ejes = plt.subplots(2, 2, figsize=(13, 11))
for fila_ejes, (nombre, kv) in zip(ejes, [("W2V noticias", w2v_mejor.wv), ("fastText SUC", suc)]):
    X = np.array([kv[w] for w in palabras_viz])
    proyecciones = {
        "PCA": PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X),
        "t-SNE": TSNE(n_components=2, perplexity=15, metric="cosine", init="pca",
                      random_state=RANDOM_STATE).fit_transform(X),
    }
    for ax, (metodo, P) in zip(fila_ejes, proyecciones.items()):
        for grupo in GRUPOS:
            m = np.array(grupo_viz) == grupo
            ax.scatter(P[m, 0], P[m, 1], s=22, color=COLORES[grupo], label=grupo)
        for i, w in enumerate(palabras_viz):
            if w in ROTULOS:
                ax.annotate(w, P[i], fontsize=8, xytext=(3, 3), textcoords="offset points")
        ax.set_title(f"{nombre} · {metodo}")
        ax.set_xticks([])
        ax.set_yticks([])
ejes[0, 0].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig("reporte/img/espacio_pca_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 6.3 ¿Qué tan separados quedan los grupos? ---
# Silueta con coseno sobre los vectores de 100 dimensiones, no sobre la proyección.
from sklearn.metrics import silhouette_score, silhouette_samples

filas = []
for nombre, kv in [("W2V noticias", w2v_mejor.wv), ("fastText SUC", suc)]:
    X = np.array([kv[w] for w in palabras_viz])
    por_palabra = silhouette_samples(X, grupo_viz, metric="cosine")
    fila = {"modelo": nombre, "total": silhouette_score(X, grupo_viz, metric="cosine")}
    fila.update({g: por_palabra[np.array(grupo_viz) == g].mean() for g in GRUPOS})
    filas.append(fila)
pd.DataFrame(filas).set_index("modelo").round(3)

### 6.4 Resultados y discusión

Usamos 150 palabras de seis grupos (33 países, 12 meses, 22 números, 23 verbos de movimiento, 31 términos económicos y 29 profesiones) que están en los dos modelos. La figura queda en `reporte/img/espacio_pca_tsne.png`.

#### ¿Se forman los grupos esperados? ¿En qué modelo se separan mejor?

En SUC sí, y muy claros. Con t-SNE salen seis islas casi sin mezcla, el único punto fuera de lugar es "partir", que cae con los meses por el "a partir de enero". En nuestro Word2Vec los países, los meses y las profesiones se agrupan bien, pero los términos económicos quedan regados por todo el mapa y se mezclan con los números y los verbos. Tiene sentido, en las noticias casi todas las palabras son de economía, así que "inflación" o "dólar" aparecen al lado de cualquier cosa. De hecho en el PCA "inflación" cae pegada a los meses, seguramente porque las noticias reportan la inflación mes a mes ("la inflación de julio").

La silueta con coseno (sobre las 100 dimensiones, no sobre el dibujo) lo confirma.

| Grupo | W2V noticias | fastText SUC |
|---|---|---|
| meses | 0.641 | 0.953 |
| números | 0.198 | 0.611 |
| economía | −0.079 | 0.302 |
| total | 0.206 | 0.418 |

SUC duplica a nuestro modelo, y economía es el único grupo con silueta negativa en el nuestro, o sea que en promedio esas palabras están más cerca de otro grupo que del suyo.

#### ¿Por qué en t-SNE se puede confiar en los grupos pero no en la distancia entre grupos?

PCA es una proyección lineal, conserva las direcciones de mayor varianza y por eso las distancias grandes se mantienen más o menos, pero los grupos se enciman (en SUC las profesiones y los verbos quedan uno encima del otro). t-SNE en cambio solo intenta que cada palabra conserve a sus vecinos cercanos, por lo tanto los grupos sí son reales, pero dónde queda cada isla respecto a las otras es bastante arbitrario y cambia con la semilla o la perplejidad. Que en SUC los números queden lejos de los meses no quiere decir que sean más distintos que los países y las profesiones.